# Set 6 오답노트

## 검토한 파일

- `set_01_06_answer/06_questions.ipynb`
- `set_01_06_answer/06_answer.ipynb`
- `00_trying/01/06_question.ipynb`
- `00_trying/02/06_questions.ipynb`
- `00_trying/02/06_questions copy.ipynb`

## 최종 답

- Q01: 경험 유무별 희망 비율의 비 `A/B` = **1.77**
- Q02: 가장 큰 Odds Ratio = **1.67** — `enrolled_university_Full time course`
- Q03: 지표 A(Accuracy) = **0.71**

`00_trying/02/06_questions.ipynb`는 Q1 작성 도중이고 Q2·Q3은 비어 있다. 복사본에는 Q1·Q2가 작성되어 있고 Q3은 시작만 되어 있으므로, 완성된 첫 풀이와 기준 답안을 함께 비교해 정리한다.


## 공통 전처리 — 19,158행에서 7,522행의 base 만들기

### 1. 사용하지 않는 열 제거

```python
base = df.drop(
    columns=['city', 'company_size', 'company_type']
).copy()
```

열은 15개에서 12개로 줄어든다. `.copy()`를 사용하면 이후 값을 변경할 때 원본과 분리되고 `SettingWithCopyWarning`을 예방할 수 있다.

### 2. 문자형 변수에 결측치가 있는 행만 제거

문제는 모든 열의 결측치를 제거하라고 하지 않고 **문자형 변수에 결측치가 하나라도 존재하는 행**을 제거하라고 했다.

```python
object_cols = base.select_dtypes(include='object').columns
base = base.dropna(subset=object_cols).copy()
```

`select_dtypes('object')`는 dtype이 object인 열 이름만 선택한다. 현재 데이터는 숫자형 분석 열에 결측치가 없어 전체 `dropna()`를 사용해도 결과가 같을 수 있지만, 다른 데이터에서는 숫자형 결측치 행까지 불필요하게 삭제하므로 문제 조건대로 `subset`을 지정한다.

### 3. 특수 문자열 행을 제거한 후 정수 변환

```python
valid_exp = ~base['experience'].isin(['>20', '<1'])
base = base.loc[valid_exp].copy()
base['experience'] = base['experience'].astype(int)

valid_job = ~base['last_new_job'].isin(['>4', 'never'])
base = base.loc[valid_job].copy()
base['last_new_job'] = base['last_new_job'].astype(int)

display(base.shape)  # (7522, 12)
```

문자열 `'>20'`, `'<1'`, `'>4'`, `'never'`가 남아 있는 상태에서는 전체 열을 정수로 변환할 수 없다. 먼저 해당 행을 제거하고 `.astype(int)`를 적용한다. `~`는 Boolean 조건을 반전하여 해당 값이 아닌 행을 남긴다.


## Q01 — 그룹별 target=1 비율과 A/B

### `value_counts(normalize=True)` 방식

```python
ratio_table = (
    base.groupby('relevant_experience')['target']
    .value_counts(normalize=True)
)
display(ratio_table)
```

실제 결과:

```text
relevant_experience      target
Has relevant experience  0.0       0.784089
                         1.0       0.215911
No relevant experience   0.0       0.617127
                         1.0       0.382873
```

결과는 `(relevant_experience, target)` 두 단계 인덱스를 가진 MultiIndex Series다. 문제의 A와 B는 다음처럼 라벨로 선택한다.

```python
A = ratio_table.loc[('No relevant experience', 1.0)]
B = ratio_table.loc[('Has relevant experience', 1.0)]
answer_q1 = round(A / B, 2)

display(A)          # 0.382873...
display(B)          # 0.215911...
display(answer_q1)  # 1.77
```

`value_counts(normalize=True)`는 각 경험 그룹 안에서 target 값별 개수를 그 그룹의 전체 개수로 나눠 비율을 반환한다. `normalize=False`가 기본값이며 이 경우 단순 개수만 나온다.

### 이진 target에서는 평균이 곧 1의 비율

target 값이 0과 1뿐이면 그룹 평균은 1의 비율과 같다.

```text
평균 = (0의 합 + 1의 합) / 전체 개수
     = 1의 개수 / 전체 개수
     = target=1의 비율
```

따라서 다음 풀이가 가장 간결하다.

```python
target_rate = base.groupby('relevant_experience')['target'].mean()
A = target_rate.loc['No relevant experience']
B = target_rate.loc['Has relevant experience']
answer_q1 = round(A / B, 2)
```

### 사용자 정의 `ratio()` 함수의 개선점

현재 작성 중인 함수는 `ser.value_counts()[1.0] / ser.shape[0]`으로 1의 비율을 계산하므로 원리는 맞다. 다만 그룹 안에 target=1이 하나도 없으면 `value_counts()[1.0]`에서 KeyError가 발생할 수 있다. 이진 데이터에서는 `ser.mean()`이 더 짧고 안전하다.

```python
def ratio(ser):
    return ser.mean()

target_rate = base.groupby('relevant_experience')['target'].apply(ratio)
```

함수 안의 `display()`는 그룹마다 출력되어 디버깅에는 도움이 되지만 최종 풀이에서는 출력이 많아지므로 제거한다.


## Q02-1 — 사전순 마지막 범주를 제외한 더미변수

### 기준 범주를 제거하는 이유

범주가 k개이면 더미변수 k개 중 하나는 나머지 더미변수와 완전히 중복되는 정보를 가진다. 상수항이 있는 회귀식에서는 완전 다중공선성을 피하기 위해 한 범주를 기준 범주로 제외한다. 문제는 임의의 범주나 첫 범주가 아니라 **사전순 마지막 범주**를 제거하라고 지정했다.

`drop_first=True`는 보통 사전순 첫 범주를 제거하므로 이 문제 조건과 다르다. 직접 마지막 열을 제거해야 한다.

### Series와 DataFrame의 `get_dummies()` 차이

```python
pd.get_dummies(base['gender'])
```

Series를 전달하면 기본 열 이름은 `Female`, `Male`, `Other`처럼 범주값만 나온다. `prefix='gender'`를 지정하면 `gender_Female`처럼 원본 열 이름이 붙는다.

```python
pd.get_dummies(base['gender'], prefix='gender')
```

반면 `pd.get_dummies(base[['gender']])`처럼 DataFrame을 전달하면 원본 열 이름을 포함한 더미 열 이름이 자동 생성된다. 현재 반복문처럼 Series 하나씩 처리할 때는 `prefix=col`을 명시하는 것이 좋다.

### 안전한 생성 코드

```python
dummy_cols = [
    'gender', 'relevant_experience', 'enrolled_university',
    'education_level', 'major_discipline'
]

dummy_frames = []

for col in dummy_cols:
    dummy = pd.get_dummies(base[col], prefix=col)
    last_col = sorted(dummy.columns)[-1]
    dummy = dummy.drop(columns=last_col)
    dummy_frames.append(dummy)

dummy_df = pd.concat(dummy_frames, axis=1)
```

현재 풀이의 `dummy.iloc[:, :-1]`도 pandas가 만든 열이 사전순으로 정렬되어 있다는 전제에서 맞다. `last_col = sorted(...)[-1]`로 제거할 열을 명시하면 의도가 더 잘 드러난다. 범주값에 해당하는 행을 삭제하면 관측 데이터가 사라지므로, 행을 삭제하는 것이 아니라 생성된 더미 **열 하나만** 제거해야 한다.

실제로 제외되는 기준 범주:

- `gender`: `Other`
- `relevant_experience`: `No relevant experience`
- `enrolled_university`: `no_enrollment`
- `education_level`: `Phd`
- `major_discipline`: `STEM`

열 이름의 공백은 모델 학습에 문제되지 않는다. 보기 좋게 밑줄로 바꾸려면 결과를 다시 대입한다.

```python
dummy_df.columns = dummy_df.columns.str.replace(' ', '_')
```

### 더미변수 개수와 job2 구성

```text
gender:                  3범주 - 1 = 2개
relevant_experience:     2범주 - 1 = 1개
enrolled_university:     3범주 - 1 = 2개
education_level:         3범주 - 1 = 2개
major_discipline:        6범주 - 1 = 5개
더미변수 합계:                       12개
연속·순서형 변수:                     4개
독립변수 합계:                       16개
target, Xgrp 추가 후 job2:            18개 열
```

```python
numeric_cols = [
    'city_development_index', 'experience',
    'last_new_job', 'training_hours'
]

job2 = pd.concat(
    [base[numeric_cols], dummy_df, base[['target', 'Xgrp']]],
    axis=1
)
display(job2.shape)  # (7522, 18)
```

`pd.concat(..., axis=1)`은 인덱스를 기준으로 옆으로 붙인다. 모든 객체가 같은 base에서 파생되어 같은 원래 인덱스를 유지하므로 정확히 정렬된다. 중간에 한 객체만 `reset_index(drop=True)`를 하면 인덱스가 달라져 결측치가 생길 수 있으므로 인덱스를 일관되게 유지한다. 빈 DataFrame에 반복해서 concat하는 것보다 DataFrame들을 리스트에 모아 마지막에 한 번 concat하는 편이 간결하고 효율적이다.


## Q02-2 — Logistic Regression과 Odds Ratio

### 모델 입력

```python
X = job2.drop(columns=['target', 'Xgrp']).copy()
y = job2['target'].copy()
display(X.shape)  # (7522, 16)

model = LogisticRegression(
    C=100000,
    max_iter=1000,
    solver='liblinear',
    random_state=123,
    fit_intercept=True
)
model.fit(X, y)
```

`fit_intercept=True`는 기본값이므로 생략해도 상수항이 포함된다. 문제는 정확한 모델 조건과 원래 변수 단위에서의 계수로 정답을 정하므로 임의로 표준화를 추가하면 계수와 Odds Ratio가 달라진다.

### `coef_`의 차원과 변수 이름 연결

이진 로지스틱 회귀에서 `model.coef_.shape`는 `(1, 16)`이다.

```python
model.coef_       # 2차원: (1, 16)
model.coef_[0]    # 1차원: (16,)
```

독립변수 순서는 X의 열 순서와 같으므로 다음처럼 Series로 연결한다.

```python
odds_ratio = pd.Series(
    np.exp(model.coef_[0]),
    index=X.columns
)
```

로지스틱 회귀계수 `beta`는 log odds의 변화량이고 `exp(beta)`가 Odds Ratio다. 다른 변수가 같을 때 해당 변수가 1 증가하면 target=1의 odds가 몇 배가 되는지를 뜻한다. 더미변수의 Odds Ratio는 제외한 기준 범주와 비교한 odds 배수다.

상수항은 `model.intercept_`에 별도로 저장되며 문제에서 제외하라고 했으므로 `coef_`만 사용한다. `exp(intercept)`는 모든 독립변수가 0일 때의 기준 odds이지 특정 변수의 Odds Ratio가 아니다.

### 최댓값과 버림 처리

```python
best_feature = odds_ratio.idxmax()
best_or = odds_ratio.max()
answer_q2 = np.floor(best_or * 100) / 100

display(best_feature)  # enrolled_university_Full time course
display(best_or)       # 1.674875898...
display(answer_q2)     # 1.67
```

문제는 반올림이 아니라 소수점 셋째 자리에서 **버림**을 요구한다. `round(1.674875, 2)`는 `1.67`이어서 이번에는 같지만, 예를 들어 `1.678`이면 반올림은 `1.68`, 버림은 `1.67`로 달라진다. Odds Ratio는 양수이므로 `np.floor(x*100)/100`으로 버림할 수 있다. 정렬은 전체 순서를 보고 싶을 때만 필요하며 최댓값 하나는 바로 `.max()`와 `.idxmax()`로 구할 수 있다.


## Q03 — K-NN, 혼동행렬, Accuracy

### 1. Xgrp 기준으로 분할

```python
train = job2.loc[job2['Xgrp'] == 'train'].copy()
test = job2.loc[job2['Xgrp'] == 'test'].copy()
```

값 하나를 비교할 때는 `==`가 가장 간단하다. `.isin(['train'])`도 같은 Boolean 결과를 반환하므로 틀리지는 않는다. 문제에서 이미 Train/Test 구분을 제공했으므로 `train_test_split()`으로 무작위 재분할하면 안 된다.

### 2. 16개 독립변수로 K-NN 학습

```python
X_train = train.drop(columns=['target', 'Xgrp']).copy()
y_train = train['target'].copy()
X_test = test.drop(columns=['target', 'Xgrp']).copy()
y_test = test['target'].copy()

model = KNeighborsClassifier(
    n_neighbors=5,
    metric='euclidean'
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

K-NN은 거리 기반이므로 실제 분석에서는 변수 스케일에 민감하다. 하지만 이 문제는 별도의 정규화를 지시하지 않았고 기준 정답도 job2 원본 변수로 계산하므로 임의로 표준화를 추가하면 결과가 달라진다. 시험에서는 명시된 절차를 그대로 따른다. `KNeighborsClassifier`는 학습 과정에서 임의 초기화를 사용하지 않아 `random_state` 매개변수가 없다.

### 3. 지표 A는 Accuracy

문제의 식은 전체 예측 중 맞은 예측의 비율이다.

```text
Accuracy = (True Positive + True Negative) / 전체 데이터
```

실제 혼동행렬:

| 실제 target / 예측 | 0 | 1 |
|---|---:|---:|
| 0 | TN = 1897 | FP = 195 |
| 1 | FN = 616 | TP = 108 |

`pd.crosstab(y_test, y_pred)`에서 행은 실제값, 열은 예측값이다. 대각선의 1897과 108이 맞게 예측한 개수이고 전체는 2816개다.

```python
confusion = pd.crosstab(y_test, y_pred)
correct = confusion.values.diagonal().sum()
total = confusion.values.sum()
answer_q3 = round(correct / total, 2)
display(answer_q3)  # 0.71
```

직접 위치로 선택하는 방법도 가능하다.

```python
confusion = confusion.reindex(index=[0.0, 1.0], columns=[0.0, 1.0], fill_value=0)
TN = confusion.loc[0.0, 0.0]
TP = confusion.loc[1.0, 1.0]
answer_q3 = round((TN + TP) / confusion.to_numpy().sum(), 2)
```

현재 데이터에서는 `.iloc[0,0]`, `.iloc[1,1]`도 맞지만, 어떤 평가 데이터에 특정 클래스가 하나도 없으면 행이나 열이 사라져 위치 기반 접근이 잘못될 수 있다. 라벨 기반 `.loc`와 `reindex()`가 더 안전하다.

가장 간단한 방법은 sklearn의 Accuracy 함수다.

```python
from sklearn.metrics import accuracy_score
answer_q3 = round(accuracy_score(y_test, y_pred), 2)
# 0.71
```

Accuracy `0.71`은 전체의 약 71%를 맞혔다는 뜻이다. 다만 target=1 표본이 적은 불균형 데이터에서는 Accuracy만으로 희망자 탐지 성능을 충분히 평가하기 어렵다. 문제는 지정된 지표 A만 요구하므로 Accuracy를 답으로 사용한다.


## 핵심 암기

1. 문자형 열 결측치 제거는 `select_dtypes('object')`로 열을 찾고 `dropna(subset=...)`을 사용한다.
2. target이 0/1이면 그룹 평균이 곧 target=1의 비율이다.
3. `value_counts(normalize=True)` 결과는 그룹값과 target의 MultiIndex Series다.
4. `drop_first=True`는 첫 범주를 제거하므로 사전순 마지막 범주 제거 문제에는 그대로 쓰면 안 된다.
5. 범주 행을 삭제하는 것이 아니라 생성한 더미 열 중 기준 범주 열 하나만 제거한다.
6. `pd.concat(axis=1)`은 인덱스로 정렬하므로 결합할 객체들의 인덱스를 일관되게 유지한다.
7. 이진 Logistic Regression의 `coef_`는 `(1, 변수개수)`이고 `np.exp(coef_[0])`가 변수별 Odds Ratio다.
8. 상수항은 `intercept_`에 별도로 있으며 문제의 Odds Ratio 비교에서 제외한다.
9. 버림과 반올림을 구분한다. 양수 둘째 자리 버림은 `floor(x*100)/100`이다.
10. 제공된 Xgrp가 있으면 그 값으로 Train/Test를 나누고 임의 재분할하지 않는다.
11. 지표 `(TP+TN)/전체`는 Accuracy이며 혼동행렬의 대각선 합을 전체 합으로 나눈 값이다.
12. K-NN은 거리 기반이고 실제 분석에서는 스케일에 민감하지만 시험에서는 제시된 전처리 절차를 따른다.
